This notebook supports exploring the partisan bias of ensembles

In [1]:
from typing import List, Dict, Any, Set

import os
import pandas as pd
from collections import defaultdict

from rdametrics import states, chambers, ensembles

Load the scores dataframe

In [2]:
scores_path: str = "~/local/beta-ensembles/prepackaged/scores/scores.parquet"
scores_df = pd.read_parquet(os.path.expanduser(scores_path))

Grab the partisan leans for each state

In [4]:
partisan_leans: Dict[str, float] = dict()
n_states: int = len(states)

for index, row in scores_df.iterrows():
    state = row["state"]
    if state in partisan_leans:
        continue

    partisan_leans[state] = row["estimated_vote_pct"]

    if len(partisan_leans) >= n_states:
        break

partisan_leans

{'FL': 0.4837,
 'IL': 0.5817,
 'MI': 0.5188,
 'NC': 0.4943,
 'NY': 0.6478,
 'OH': 0.4638,
 'WI': 0.5068}

In [5]:
sorted(partisan_leans.items(), key=lambda x: x[1])

[('OH', 0.4638),
 ('FL', 0.4837),
 ('NC', 0.4943),
 ('WI', 0.5068),
 ('MI', 0.5188),
 ('IL', 0.5817),
 ('NY', 0.6478)]

Count various quantities of interest

In [7]:
from rdapy import read_json

table_dir: str = (
    "~/Documents/work/Ensembles/partisan-bias-of-ensembles/tables/intermediate"
)

table_files = ("bias_tables.json", "zero_tables.json")

table1_path = os.path.expanduser(os.path.join(table_dir, table_files[0]))
table1: dict = read_json(table1_path)

favor_dems: int = 0
total: int = 0

for variant, _data in table1.items():
    for combo, _measures in _data.items():
        for m, value in _measures.items():
            total += 1
            if value is not None and value < 0:
                favor_dems += 1

print(f"{favor_dems} / {total} ({favor_dems/total:.2%}) favor Democrats")
print(f"{total - favor_dems} / {total} ({(total - favor_dems)/total:.2%}) favor Republicans")

116 / 1848 (6.28%) favor Democrats
1732 / 1848 (93.72%) favor Republicans


In [12]:
from rdapy import read_json

def bucket_zeroes(values):
    counts = {'A': 0, 'B': 0, 'C': 0, 'D': 0}
    
    for val in values:
        if val == 1.0:
            counts['A'] += 1
        elif val >= 0.95:
            counts['B'] += 1
        elif val > 0.6:
            counts['C'] += 1
        else:
            counts['D'] += 1
    
    return counts

table_dir: str = (
    "~/Documents/work/Ensembles/partisan-bias-of-ensembles/tables/intermediate"
)

table_files = ("bias_tables.json", "zero_tables.json")

table1_path = os.path.expanduser(os.path.join(table_dir, table_files[0]))
table2_path = os.path.expanduser(os.path.join(table_dir, table_files[1]))

table1: dict = read_json(table1_path)
table2: dict = read_json(table2_path)

inverted_zeroes: List[float] = list()

for variant, _data in table1.items():
    for combo, _measures in _data.items():
        for m, value in _measures.items():
            z = (
                table2[variant][combo][m]
                if value < 0
                else 1 - table2[variant][combo][m]
            )
            inverted_zeroes.append(z)

buckets = {k: f"{v / len(inverted_zeroes):.2%}" for k, v in bucket_zeroes(inverted_zeroes).items()}

buckets

{'A': '45.73%', 'B': '34.42%', 'C': '17.91%', 'D': '1.95%'}